In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import random
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
import pprint
import pyspark
import pyspark.sql.functions as F
import utils
from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, FloatType, DateType

import utils.data_processing_bronze_table
import utils.data_processing_attribute_silver_table
import utils.data_processing_financials_silver_table
import utils.data_processing_clickstream_silver_table
import utils.data_processing_lms_silver_table
import utils.data_processing_lms_gold_table

In [2]:
# Initialize SparkSession
spark = (
    pyspark.sql.SparkSession.builder
.appName("dev")
.master("local[*]")
.getOrCreate()
)

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

# star is to use all available resources on the local machine. You can also specify a number, e.g., "local[4]" to use 4 cores.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 07:32:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
gold_label_store_directory = "datamart/gold/label_store/"

if not os.path.exists(gold_label_store_directory):
    os.makedirs(gold_label_store_directory)

In [4]:
snapshot_date_str = "2023-01-01"

start_date_str = "2023-01-01"
end_date_str = "2024-12-01"

def generate_first_of_month_dates(start_date_str, end_date_str):
    # Convert the date strings to datetime objects
    start_date = datetime.strptime(start_date_str, "%Y-%m-%d")
    end_date = datetime.strptime(end_date_str, "%Y-%m-%d")
    
    # List to store the first of month dates
    first_of_month_dates = []

    # Start from the first of the month of the start_date
    current_date = datetime(start_date.year, start_date.month, 1)

    while current_date <= end_date:
        # Append the date in yyyy-mm-dd format
        first_of_month_dates.append(current_date.strftime("%Y-%m-%d"))
        
        # Move to the first of the next month
        if current_date.month == 12:
            current_date = datetime(current_date.year + 1, 1, 1)
        else:
            current_date = datetime(current_date.year, current_date.month + 1, 1)

    return first_of_month_dates

dates_str_lst = generate_first_of_month_dates(start_date_str, end_date_str)
dates_str_lst

['2023-01-01',
 '2023-02-01',
 '2023-03-01',
 '2023-04-01',
 '2023-05-01',
 '2023-06-01',
 '2023-07-01',
 '2023-08-01',
 '2023-09-01',
 '2023-10-01',
 '2023-11-01',
 '2023-12-01',
 '2024-01-01',
 '2024-02-01',
 '2024-03-01',
 '2024-04-01',
 '2024-05-01',
 '2024-06-01',
 '2024-07-01',
 '2024-08-01',
 '2024-09-01',
 '2024-10-01',
 '2024-11-01',
 '2024-12-01']

In [7]:
silver_loan_daily_directory = "datamart/silver/lms_loan_daily/"
for date_str in dates_str_lst:
    utils.data_processing_lms_gold_table.process_labels_gold_table(date_str, silver_loan_daily_directory, gold_label_store_directory, spark, dpd = 30, mob = 6)


loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_01_01.parquet row count: 530


saved to: datamart/gold/label_store/gold_label_store_2023_01_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_02_01.parquet row count: 1031


saved to: datamart/gold/label_store/gold_label_store_2023_02_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_03_01.parquet row count: 1537
saved to: datamart/gold/label_store/gold_label_store_2023_03_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_04_01.parquet row count: 2047


saved to: datamart/gold/label_store/gold_label_store_2023_04_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_05_01.parquet row count: 2568


saved to: datamart/gold/label_store/gold_label_store_2023_05_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2023_06_01.parquet row count: 3085


...

saved to: datamart/gold/label_store/gold_label_store_2024_06_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_07_01.parquet row count: 5442


saved to: datamart/gold/label_store/gold_label_store_2024_07_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_08_01.parquet row count: 5531
saved to: datamart/gold/label_store/gold_label_store_2024_08_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_09_01.parquet row count: 5537


saved to: datamart/gold/label_store/gold_label_store_2024_09_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_10_01.parquet row count: 5502
saved to: datamart/gold/label_store/gold_label_store_2024_10_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_11_01.parquet row count: 5501


saved to: datamart/gold/label_store/gold_label_store_2024_11_01.parquet
loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_2024_12_01.parquet row count: 5531


saved to: datamart/gold/label_store/gold_label_store_2024_12_01.parquet


In [5]:
gold_label_store_directory = "datamart/gold/label_store/"
silver_loan_daily_directory = "datamart/silver/lms_loan_daily/"
# gold_label_main = utils.data_processing_lms_gold_table.process_labels_gold_table_main(silver_loan_daily_directory,gold_label_store_directory,spark,dpd = 30, mob = 6)

loaded from: datamart/silver/lms_loan_daily/silver_loan_daily_main_.parquet row count: 137500


saved to: datamart/gold/label_store/gold_label_store_main_.parquet


In [3]:
## setting up file directory config:
bronze_directory = "datamart/bronze/"

attribute_filename = "features_attributes/features_attributes_main.csv"
financial_filename = "features_financials/features_financials_main.csv"
loan_filename = "lms_loan_daily/lms_loan_daily_main.csv"
clickstream_filename = "feature_clickstream/feature_clickstream_main.csv"

silver_attributes_daily_directory = "datamart/silver/features_attributes/"
silver_financials_daily_directory = "datamart/silver/features_financials/"
silver_loan_daily_directory = "datamart/silver/lms_loan_daily/"
silver_clickstream_directory ="datamart/silver/feature_clickstream/"

silver_attribute_filename = "features_attributes/silver_features_attributes_main.parquet"
silver_financial_filename = "features_financials/silver_features_financials_main.parquet"
gold_loan_filename = "datamart/gold/label_store/gold_label_store_main_.parquet"
silver_clickstream_filename = "feature_clickstream/silver_clickstream_main_.parquet"


silver_directory = "datamart/silver/"
os.makedirs(silver_directory, exist_ok = True)

gold_directory = "datamart/gold/"
os.makedirs(gold_directory, exist_ok = True)

silver_merged_processed_directory = "datamart/silver/merged/"
os.makedirs(silver_merged_processed_directory, exist_ok = True)

gold_features_main_directory = "datamart/gold/feature_store/"
os.makedirs(gold_features_main_directory, exist_ok = True)


os.makedirs(bronze_directory, exist_ok = True)
os.makedirs(silver_attributes_daily_directory, exist_ok = True)
os.makedirs(silver_financials_daily_directory, exist_ok = True)
os.makedirs(silver_loan_daily_directory, exist_ok = True)
os.makedirs(silver_clickstream_directory, exist_ok = True)

data_start_date_str = "2023-01-01"
data_end_date_str = "2024-12-01"

train_end_date = '2024-09-01'
unseen_start_date = '2024-10-01'


In [4]:
utils.data_processing_lms_gold_table.process_features_gold_table_main(silver_directory, spark, 
                            silver_attribute_filename, silver_financial_filename, gold_loan_filename, silver_clickstream_filename,gold_features_main_directory,
                            train_end_date, unseen_start_date,data_start_date_str,data_end_date_str,
                            model_train_date_str="2025-06-01", validation_period_months=1,
                            test_period_months=2, oot_period_months=2)

loaded from: datamart/silver/features_attributes/silver_features_attributes_main.parquet row count: 12500
loaded from: datamart/silver/features_financials/silver_features_financials_main.parquet row count: 12500
loaded from: datamart/gold/label_store/gold_label_store_main_.parquet row count: 12500
loaded from: datamart/silver/feature_clickstream/silver_clickstream_main_.parquet row count: 215376

Date split boundaries:
Train: 2023-01-01 to 2024-12-31
Validation: 2025-01-01 to 2025-01-31
Test: 2025-02-01 to 2025-03-31
OOT: 2025-04-01 to 2025-05-31



Train: 8974 rows
+-----+-----+-------------------+
|label|count|         proportion|
+-----+-----+-------------------+
|    0| 6383| 0.7112770225094718|
|    1| 2591|0.28872297749052817|
+-----+-----+-------------------+

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2023-07-01|2024-12-01|
+----------+----------+


Validation: 505 rows
+-----+-----+------------------+
|label|count|        proportion|
+-----+-----+------------------+
|    0|  350| 0.693069306930693|
|    1|  155|0.3069306930693069|
+-----+-----+------------------+

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-01-01|2025-01-01|
+----------+----------+


Test: 1036 rows
+-----+-----+-------------------+
|label|count|         proportion|
+-----+-----+-------------------+
|    0|  710| 0.6853281853281853|
|    1|  326|0.31467181467181465|
+-----+-----+-------------------+

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-02-01|2025-03-

+------------+-------+------------------+----------+-----+
|  Occupation|Age_Bin| log_income_median|age_median|count|
+------------+-------+------------------+----------+-----+
|MediaManager|  35-42|10.482973805102581|        38|  134|
|  Accountant|  27-34|10.487246912994182|        30|  123|
|  Accountant|  18-26|10.534979551138033|        23|  124|
|     Teacher|  27-34|10.431097466232934|        31|  121|
|    Engineer|  27-34|10.547302713164944|        31|  133|
|   Scientist|  43-56|10.649030090887043|        47|  134|
|Entrepreneur|  18-26|10.461044078789724|        22|  128|
|            |  27-34|10.473715360069058|        31|  157|
|  Journalist|  35-42|10.353238088050679|        38|  128|
|MediaManager|  43-56|10.566129616052383|        48|  134|
|    Mechanic|  43-56|10.787507917792388|        48|  127|
|  Journalist|  27-34|10.521021256243461|        31|  121|
|      Doctor|  18-26|10.545569311592208|        22|  141|
|            |  18-26|10.518878618160022|        22|  13

+-----------+-----------------+-------------------+---------------------+-------------------------+-------------------------+-------------------------------+---------------+------------------+--------------------+------------------------+------------------------+------------------------------+
|Customer_ID|Num_Bank_Accounts|Bank_Accounts_clean|Bank_Accounts_imputed|Bank_Accounts_Imputed_Bin|Bank_Accounts_Was_Invalid|Bank_Accounts_imputation_method|Num_Credit_Card|Credit_Cards_clean|Credit_Cards_imputed|Credit_Cards_Imputed_Bin|Credit_Cards_Was_Invalid|Credit_Cards_imputation_method|
+-----------+-----------------+-------------------+---------------------+-------------------------+-------------------------+-------------------------------+---------------+------------------+--------------------+------------------------+------------------------+------------------------------+
|CUS_0x69cd |5                |5                  |5                    |4-5                      |0               

+-----------+-----------------+-------------------+---------------------+-------------------------+-------------------------+-------------------------------+---------------+------------------+--------------------+------------------------+------------------------+------------------------------+
|Customer_ID|Num_Bank_Accounts|Bank_Accounts_clean|Bank_Accounts_imputed|Bank_Accounts_Imputed_Bin|Bank_Accounts_Was_Invalid|Bank_Accounts_imputation_method|Num_Credit_Card|Credit_Cards_clean|Credit_Cards_imputed|Credit_Cards_Imputed_Bin|Credit_Cards_Was_Invalid|Credit_Cards_imputation_method|
+-----------+-----------------+-------------------+---------------------+-------------------------+-------------------------+-------------------------------+---------------+------------------+--------------------+------------------------+------------------------+------------------------------+
|CUS_0x559e |1665             |NULL               |6                    |6-7                      |1               

Length of temp after LEFT join: 215376


Length of filtered temp (days_before > 0): 129778


Feature rows after clickstream enrichment: 11459
Clickstream columns joined to gold table: ['labels_snapshot_date', 'fe_10_mean_90d', 'fe_5_mean_90d', 'fe_4_mean_90d', 'fe_9_mean_90d', 'fe_3_std_60d', 'fe_8_mean_90d', 'fe_3_mean_90d', 'fe_14_trend_30d', 'fe_1_trend_30d', 'fe_16_std_60d']


+-----------+--------------------+--------------------+--------------+-------------+------------------+------------------+------------+------------------+------------------+---------------+--------------+-------------+
|Customer_ID|labels_snapshot_date|labels_snapshot_date|fe_10_mean_90d|fe_5_mean_90d|fe_4_mean_90d     |fe_9_mean_90d     |fe_3_std_60d|fe_8_mean_90d     |fe_3_mean_90d     |fe_14_trend_30d|fe_1_trend_30d|fe_16_std_60d|
+-----------+--------------------+--------------------+--------------+-------------+------------------+------------------+------------+------------------+------------------+---------------+--------------+-------------+
|CUS_0x5035 |2024-06-01          |2024-06-01          |229.0         |42.0         |153.5             |130.0             |NULL        |121.0             |85.5              |NULL           |NULL          |NULL         |
|CUS_0x5036 |2023-10-01          |2023-10-01          |77.0          |131.5        |-62.5             |191.0             |NU

saved to: datamart/gold/feature_store/gold_feature_store_main_cs.parquet


In [5]:
fs_sdf_cs = spark.read.option("header", "true").parquet('datamart/gold/feature_store/gold_feature_store_main_cs.parquet')
fs_sdf_cs = fs_sdf_cs.toPandas()
display(fs_sdf_cs.head())
display(fs_sdf_cs.info())
print("number of customers:", fs_sdf_cs['Customer_ID'].nunique())

,Customer_ID,labels_snapshot_date,Occupation,Age_Bin,loan_id,Age,attributes_snapshot_date,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,fe_10_mean_90d,fe_5_mean_90d,fe_4_mean_90d,fe_9_mean_90d,fe_3_std_60d,fe_8_mean_90d,fe_3_mean_90d,fe_14_trend_30d,fe_1_trend_30d,fe_16_std_60d
0,CUS_0x2174,2023-08-01,Manager,36-44,CUS_0x2174_2023_02_01,36,2023-02-01,94934.937500,8039.245117,2,...,170.500000,66.0,-55.500000,3.000000,NaN,141.500000,188.5,NaN,NaN,NaN
1,CUS_0x78eb,2024-05-01,Musician,36-44,CUS_0x78eb_2023_11_01,38,2023-11-01,131121.750000,11211.812500,4,...,176.666667,168.0,47.333333,112.333333,NaN,147.666667,162.0,NaN,NaN,NaN
2,CUS_0x8bcc,2023-11-01,,45-56,CUS_0x8bcc_2023_05_01,51,2023-05-01,123485.039062,10299.419922,4,...,106.500000,42.5,171.000000,121.000000,NaN,-32.000000,-73.5,NaN,NaN,NaN
3,CUS_0x6f40,2024-04-01,Musician,31-35,CUS_0x6f40_2023_10_01,31,2023-10-01,82995.820312,7092.318359,2,...,37.000000,8.5,4.000000,145.500000,12.727922,-9.500000,157.0,NaN,NaN,65.053824
4,CUS_0x19a7,2024-07-01,Musician,31-35,CUS_0x19a7_2024_01_01,34,2024-01-01,62348.460938,4942.705078,4,...,154.000000,140.0,120.000000,108.500000,NaN,255.000000,167.0,NaN,NaN,NaN


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11459 entries, 0 to 11458
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   11459 non-null  object 
 1   labels_snapshot_date          11459 non-null  object 
 2   Occupation                    11459 non-null  object 
 3   Age_Bin                       11459 non-null  object 
 4   loan_id                       11459 non-null  object 
 5   Age                           11459 non-null  int32  
 6   attributes_snapshot_date      11459 non-null  object 
 7   Annual_Income                 11459 non-null  float32
 8   Monthly_Inhand_Salary         11459 non-null  float32
 9   Num_Bank_Accounts             11459 non-null  int32  
 10  Num_Credit_Card               11459 non-null  int32  
 11  Interest_Rate                 11459 non-null  int32  
 12  Delay_from_due_date           11459 non-null  int32  
 13  N

None

number of customers: 11459


In [21]:
fs_sdf = spark.read.option("header", "true").parquet('datamart/gold/feature_store/gold_feature_store_main_.parquet')
fs_sdf = fs_sdf.toPandas()
display(fs_sdf.head())
display(fs_pdf.info())
print("number of customers:", fs_pdf['Customer_ID'].nunique())

,Occupation,Age_Bin,Customer_ID,loan_id,labels_snapshot_date,Age,attributes_snapshot_date,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,fe_20,clicks_snapshot_date,Age_imputed,Bank_Accounts_imputed,Credit_Cards_imputed,Bank_Accounts_Imputed_Bin,Credit_Cards_Imputed_Bin,Debt_per_Credit_Account,Delayed_Payments_per_Card,InterestRate_x_Debt
0,Developer,37-43,CUS_0x7409,CUS_0x7409_2023_11_01,2024-05-01,38,2023-11-01,29510.869141,2560.239258,4,...,74.0,2024-05-01,38,4,7,4-5,6-7,104.329090,-0.285714,13771.439453
1,Developer,37-43,CUS_0x706d,CUS_0x706d_2023_04_01,2023-10-01,39,2023-04-01,62030.640625,4883.220215,3,...,59.0,2023-10-01,39,3,9,0-3,8-10,142.160838,1.666667,42648.250000
2,Developer,37-43,CUS_0x6b44,CUS_0x6b44_2023_12_01,2024-06-01,39,2023-12-01,20742.460938,1801.538330,5,...,-78.0,2024-06-01,39,5,4,4-5,0-4,146.652222,4.500000,10558.959961
3,Developer,37-43,CUS_0x7693,CUS_0x7693_2023_06_01,2023-12-01,40,2023-06-01,52958.429688,4313.202637,8,...,182.0,2023-12-01,40,8,5,8-10,5-5,NaN,NaN,53673.281250
4,Developer,37-43,CUS_0x6dec,CUS_0x6dec_2023_02_01,2023-08-01,42,2023-02-01,16511.394531,1528.949585,2,...,0.0,2023-08-01,42,2,5,0-3,5-5,172.295707,0.000000,9648.559570


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9479 entries, 0 to 9478
Data columns (total 61 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Customer_ID                   9479 non-null   object 
 1   labels_snapshot_date          9479 non-null   object 
 2   Occupation                    9479 non-null   object 
 3   Age_Bin                       9479 non-null   object 
 4   loan_id                       9479 non-null   object 
 5   Age                           9479 non-null   int32  
 6   attributes_snapshot_date      9479 non-null   object 
 7   Annual_Income                 9479 non-null   float32
 8   Monthly_Inhand_Salary         9479 non-null   float32
 9   Num_Bank_Accounts             9479 non-null   int32  
 10  Num_Credit_Card               9479 non-null   int32  
 11  Interest_Rate                 9479 non-null   int32  
 12  Delay_from_due_date           9479 non-null   int32  
 13  Num

None

number of customers: 9479


In [16]:
fs_pdf['Customer_ID'].duplicated().sum()

np.int64(0)